## Powstawanie i rozrost wiru cyklonicznego – płytka woda z siłą Coriolisa (PyMPDATA)
Notebook rozszerza solwer z `lab33.ipynb` o: **siłę Coriolisa**, **tarcie o dno** i **źródło/ujście masy**.
Ujście masy w środku domeny to zabawkowy odpowiednik „niżu”: woda dopływa do środka, Coriolis ją zakręca i powstaje wir, który z czasem rośnie.

**Uwaga:** to model wirów synoptycznych w jednej warstwie płynu, a nie model huraganu (brak konwekcji i ciepła utajonego).

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PyMPDATA')

In [ ]:
import os
os.environ["NUMBA_THREADING_LAYER"]='workqueue'
import numpy as np
from matplotlib import pyplot
from open_atmos_jupyter_utils import show_plot, show_anim
from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions

### Równania (jednostki bezwymiarowe)
$$
\begin{cases}
\partial_t h + \nabla\cdot(\vec u h) = Q(x,y)\\
\partial_t (hu) + \nabla\cdot(\vec u hu) = -gh\,\partial_x h + f\,hv - r\,hu\\
\partial_t (hv) + \nabla\cdot(\vec u hv) = -gh\,\partial_y h - f\,hu - r\,hv
\end{cases}
$$
- $f$ – parametr Coriolisa (f-płaszczyzna, stały), $r$ – współczynnik tarcia Rayleigha,
- $Q<0$ w centrum (ujście masy), skompensowane jednorodnym źródłem tak, żeby całkowita masa się zachowywała,
- promień Rossby'ego: $R_d=\sqrt{gH}/f$.

Człony źródłowe liczone są w półkrokach jak w oryginalnym notebooku: gradient ciśnienia, Coriolis jako dokładny obrót pary $(hu,hv)$ o kąt $f\,\Delta t/2$, tarcie jako $e^{-r\Delta t/2}$.

### Solwer

In [ ]:
class CyclogenesisSolver:
    """Płytka woda + Coriolis + tarcie + źródło/ujście masy (MPDATA, periodyczne BC)."""
    def __init__(self, *, h_initial, options=None):
        options = options or Options(nonoscillatory=True, infinite_gauge=True)
        grid = h_initial.shape
        stepper = Stepper(options=options, grid=grid)
        kwargs = {
            'boundary_conditions': [boundary_conditions.Periodic()] * 2, 
            'halo': options.n_halo
            }
        advectees = {
            "h":  ScalarField(h_initial.astype(float), **kwargs),
            "uh": ScalarField(np.zeros(grid), **kwargs),
            "vh": ScalarField(np.zeros(grid), **kwargs),
        }
        self.advector = VectorField((np.zeros((grid[0]+1, grid[1])),
                                     np.zeros((grid[0], grid[1]+1))
                                     ),**kwargs)
        self.solvers = {k: Solver(stepper, v, self.advector) for k, v in advectees.items()}

    def __getitem__(self, key):
        return self.solvers[key].advectee.get()

    def _update_courant_numbers(self, dt_over_dxy, eps):
        h = np.maximum(self['h'], eps)
        for axis, key in enumerate(("uh", "vh")):
            vel = self[key] / h
            face = 0.5 * (vel + np.roll(vel, 1, axis=axis))   # prędkość na ścianie i-1/2 (periodycznie)
            comp = self.advector.get_component(axis)
            n = vel.shape[axis]
            if axis == 0:
                comp[:n, :] = face * dt_over_dxy[0]; comp[n, :] = comp[0, :]
            else:
                comp[:, :n] = face * dt_over_dxy[1]; comp[:, n] = comp[:, 0]
            assert np.amax(np.abs(comp)) <= 1, "liczba Couranta > 1 - zmniejsz dt"

    def _apply_half_sources(self, *, g, f, r, dt, dxy):
        h, uh, vh = self['h'], self['uh'], self['vh']
        dzdx = (np.roll(h, -1, 0) - np.roll(h, 1, 0)) / (2 * dxy[0])
        dzdy = (np.roll(h, -1, 1) - np.roll(h, 1, 1)) / (2 * dxy[1])
        uh -= 0.5 * dt * g * h * dzdx                  # gradient ciśnienia (dno płaskie: zeta = h - H)
        vh -= 0.5 * dt * g * h * dzdy
        th = f * 0.5 * dt                              # Coriolis: obrót o f*dt/2
        c, s = np.cos(th), np.sin(th)
        u_new, v_new = c * uh + s * vh, -s * uh + c * vh
        damp = np.exp(-r * 0.5 * dt)                   # tarcie Rayleigha
        uh[:] = u_new * damp
        vh[:] = v_new * damp

    def _apply_mass_source(self, Q, dt):
        """ujście zabiera pęd proporcjonalnie (nie zmienia u), źródło wnosi masę w spoczynku"""
        h = self['h']
        h_new = np.maximum(h + Q * dt, 1e-6)
        ratio = np.where(Q < 0, h_new / h, 1.0)
        self['uh'][:] *= ratio
        self['vh'][:] *= ratio
        h[:] = h_new

    def __call__(self, *, nt, g, f, r, dt, dxy=(1., 1.), Q=None, outfreq=10, eps=1e-7):
        dt_over_dxy = (dt / dxy[0], dt / dxy[1])
        out = {k: [] for k in self.solvers}
        out['t'] = []
        for it in range(nt + 1):
            if it != 0:
                self._update_courant_numbers(dt_over_dxy, eps)
                self.solvers["h"].advance(n_steps=1)
                if Q is not None:
                    self._apply_mass_source(Q, dt)
                self._apply_half_sources(g=g, f=f, r=r, dt=dt, dxy=dxy)
                self.solvers["uh"].advance(n_steps=1)
                self.solvers["vh"].advance(n_steps=1)
                self._apply_half_sources(g=g, f=f, r=r, dt=dt, dxy=dxy)
            if it % outfreq == 0:
                for k in self.solvers: out[k].append(self[k].copy())
                out['t'].append(it * dt)
        return out

### Parametry i symulacja
Jednostki: $g=H=1$ (prędkość fal $c=1$), $\Delta x=1$. Przy $f=0.1$ promień Rossby'ego to $R_d=10$ komórek.

In [ ]:
N, H, g = 64, 1.0, 1.0
f, r, dt = 0.05, 0.06, 0.1          # Coriolis, tarcie, krok czasowy
Rd = np.sqrt(g*H)/np.abs(f)
print(f"R_d = {Rd:.4f} komórek siatki")

xi, yi = np.indices((N, N))
dx_c, dy_c = xi - N/2 + .5, yi - N/2 + .5
rad = np.hypot(dx_c, dy_c)

sigma, q0 = 6.0, 0.02               # szerokość i siła ujścia masy
Q = -q0*np.exp(-rad**2/(2*sigma**2))
Q -= Q.mean()                        # zachowanie całkowitej masy

NT, OUTFREQ = 1000, 100
output = CyclogenesisSolver(h_initial=np.full((N, N), H))(
    nt=NT, g=g, f=f, r=r, dt=dt, Q=Q, outfreq=OUTFREQ
)
print("masa początkowa / końcowa:", output['h'][0].sum(), output['h'][-1].sum())

### Animacja: wir rośnie
Kolor – odchylenie powierzchni od stanu spoczynkowego $h-H$, strzałki – wiatr.

In [ ]:
hmin = min((h - H).min() for h in output['h'])
umax = max(np.hypot(u/h, v/h).max() for u, v, h in zip(output['uh'], output['vh'], output['h']))
S = 4

def plot(i):
    h = output['h'][i]; u = output['uh'][i]/h; v = output['vh'][i]/h
    fig, ax = pyplot.subplots(figsize=(6.5, 5.5))
    im = ax.imshow((h - H).T, origin='lower', cmap='RdBu', vmin=hmin, vmax=-hmin)
    ax.quiver(xi[::S, ::S], yi[::S, ::S], u[::S, ::S], v[::S, ::S],
              scale=umax*25, color='k', width=.003)
   # ax.contour(rad.T, levels=[Rd], colors='green', linestyles='--', linewidths=.8)
    ax.set_title(f"t = {output['t'][i]:.0f}")
    pyplot.colorbar(im, ax=ax, label='h - H')
    return fig

show_anim(plot, range(len(output['h'])))

---
## Porównanie z rozwiązaniem analitycznym i analiza zbieżności

### 1. Czy istnieje rozwiązanie analityczne, z którym można się porównać?

**Dla pełnego modelu – nie (w postaci zamkniętej).** Człony adwekcyjne $\nabla\!\cdot(\vec u\,hu)$, iloczyny $h\,\nabla h$, tarcie, Coriolis i gaussowskie źródło $Q(x,y)$ tworzą nieliniowy układ PDE, dla którego nie znamy rozwiązania ogólnego – dlatego go rozwiązujemy numerycznie.

**Dla modelu zlinearyzowanego – tak, i to dokładne.** Przyjmujemy $h=H+\eta$, $|\eta|\ll H$, małe prędkości (liczba Rossby'ego $\mathrm{Ro}=U/(fL)\ll 1$) i odrzucamy wyrazy nieliniowe:
$$
\partial_t\eta=-H(\partial_x u+\partial_y v)+Q,\qquad
\partial_t u=-g\,\partial_x\eta+f v-r u,\qquad
\partial_t v=-g\,\partial_y\eta-f u-r v .
$$
Układ ma stałe współczynniki, a warunki brzegowe są periodyczne, więc rozwijamy w szereg Fouriera $\propto e^{i(k_xx+k_yy)}$. Dla każdej pary $(k_x,k_y)$ dostajemy układ trzech równań zwyczajnych:
$$
\frac{d\hat{\mathbf X}}{dt}=A\hat{\mathbf X}+\hat{\mathbf b},\qquad
\hat{\mathbf X}=\begin{pmatrix}\hat\eta\\ \hat u\\ \hat v\end{pmatrix},\quad
A=\begin{pmatrix}0&-iHk_x&-iHk_y\\ -igk_x&-r&f\\ -igk_y&-f&-r\end{pmatrix},\quad
\hat{\mathbf b}=\begin{pmatrix}\hat Q\\0\\0\end{pmatrix}.
$$
Dla warunku początkowego $\hat{\mathbf X}(0)=0$ (spoczynek, $h=H$):
$$
\hat{\mathbf X}(t)=A^{-1}\!\left(e^{At}-I\right)\hat{\mathbf b}
=V\,\mathrm{diag}\!\left(\frac{e^{\lambda_j t}-1}{\lambda_j}\right)V^{-1}\hat{\mathbf b},
\qquad A=V\Lambda V^{-1},
$$
a rozwiązanie w przestrzeni fizycznej dostajemy odwrotną transformatą Fouriera. Wartości własne to pierwiastki równania $\lambda[(\lambda+r)^2+f^2]+gH|\mathbf k|^2(\lambda+r)=0$; dla $r>0$ wszystkie mają ujemną część rzeczywistą (tłumione tarciem), więc istnieje też **stan stacjonarny** $\hat{\mathbf X}_\infty=-A^{-1}\hat{\mathbf b}$ (dla $\mathbf k\neq 0$).

Rozwiązanie jest „analityczne” w tym sensie, że nie ma w nim żadnego kroku czasowego ani różnic skończonych – jest dokładne dla zlinearyzowanych równań (do błędów zaokrągleń) i spektralnie dokładne w przestrzeni.

### Skąd mogą się brać różnice numeryka–analityka?
1. **dyskretyzacja przestrzenna** – zależy od $\Delta x=L/N$,
2. **krok czasowy** $\Delta t$ (rozszczepienie operatorów w solwerze),
3. **nieliniowość** – to różnica *modeli*, a nie błąd schematu: solwer rozwiązuje pełne równania, a wzór analityczny – liniowe. Znika dla małych amplitud (małe $q_0$).

Żeby je rozdzielić, w dalszej części (a) porównujemy przypadek bazowy, (b) w reżimie liniowym (małe $q_0$) zmieniamy $N$ i $\Delta t$, (c) na koniec zmieniamy amplitudę $q_0$.

In [ ]:
import time

# parametry fizyczne = te z komórki z symulacją (dx = 1  =>  L = N)
P = dict(L=float(N), H=H, g=g, f=f, r=r, sigma=sigma, q0=q0, T=NT*dt)
c0 = np.sqrt(P['g']*P['H'])
print({k: round(v, 4) for k, v in P.items()}, f"| R_d = {c0/abs(P['f']):.2f},  1/r = {1/P['r']:.1f}")


def make_Q(Ng, q0_amp):
    """To samo ujście co w symulacji (gaussowskie, średnia odjęta), ale na siatce Ng x Ng w tej samej fizycznej domenie L x L."""
    dx = P['L'] / Ng
    X = (np.arange(Ng) + 0.5) * dx - P['L'] / 2          # środki komórek, środek domeny w (0,0)
    R2 = X[:, None]**2 + X[None, :]**2
    Qf = -q0_amp * np.exp(-R2 / (2 * P['sigma']**2))
    return Qf - Qf.mean(), dx, X


class LinearAnalytic:
    """Dokładne rozwiązanie zlinearyzowanych równań (płytka woda + Coriolis + tarcie + Q), periodyczne BC.
    Dla każdej fali Fouriera: dX/dt = A X + b,  X(0)=0  =>  X(t) = V diag((exp(lam t)-1)/lam) V^-1 b."""
    def __init__(self, Qf, dx):
        H_, g_, f_, r_ = (P[k] for k in ('H', 'g', 'f', 'r'))
        Ng = Qf.shape[0]
        k1 = 2 * np.pi * np.fft.fftfreq(Ng, d=dx)
        kx, ky = np.meshgrid(k1, k1, indexing='ij')
        A = np.zeros((Ng, Ng, 3, 3), complex)
        A[..., 0, 1], A[..., 0, 2] = -1j*H_*kx, -1j*H_*ky
        A[..., 1, 0], A[..., 1, 1], A[..., 1, 2] = -1j*g_*kx, -r_, f_
        A[..., 2, 0], A[..., 2, 1], A[..., 2, 2] = -1j*g_*ky, -f_, -r_
        b = np.zeros((Ng, Ng, 3), complex)
        b[..., 0] = np.fft.fft2(Qf)
        self.A, self.b = A, b
        self.lam, self.V = np.linalg.eig(A)
        self.coef = np.linalg.solve(self.V, b[..., None])[..., 0]    # b w bazie wektorów własnych

    @staticmethod
    def _to_space(Xh):
        X = np.fft.ifft2(Xh, axes=(0, 1)).real
        return X[..., 0], X[..., 1], X[..., 2]                       # eta, u, v

    def __call__(self, t):
        small = np.abs(self.lam) < 1e-12                             # lam = 0 (k = 0): granica = t
        phi = np.where(small, t, np.expm1(self.lam * t) / np.where(small, 1.0, self.lam))
        return self._to_space(np.einsum('xyij,xyj->xyi', self.V, phi * self.coef))

    def steady(self):
        """stan stacjonarny t -> oo:  X = -A^-1 b  (tryb k=0 ma b=0, więc X=0)"""
        A = self.A.copy()
        A[0, 0] = np.eye(3)
        return self._to_space(np.linalg.solve(A, -self.b[..., None])[..., 0])


def run_and_compare(Ng, dt_, q0_amp, T=None, every=20.0):
    """Uruchamia solwer na siatce Ng x Ng z krokiem dt_ i porównuje z rozwiązaniem analitycznym w chwilach t = every, 2*every, ... T."""
    T = P['T'] if T is None else T
    Qf, dx, X = make_Q(Ng, q0_amp)
    out = CyclogenesisSolver(h_initial=np.full((Ng, Ng), P['H']))(
        nt=int(round(T / dt_)), g=P['g'], f=P['f'], r=P['r'], dt=dt_, dxy=(dx, dx), Q=Qf,
        outfreq=int(round(every / dt_)))
    ana = LinearAnalytic(Qf, dx)
    res = dict(t=[], eta_L2=[], vel_L2=[], eta_Linf=[], X=X, dx=dx, dt=dt_, N=Ng, q0=q0_amp)
    for t, h, uh, vh in zip(out['t'], out['h'], out['uh'], out['vh']):
        if t == 0:
            continue
        ea, ua, va = ana(t)
        en, un, vn = h - P['H'], uh / h, vh / h
        res['t'].append(t)
        res['eta_L2'].append(np.linalg.norm(en - ea) / np.linalg.norm(ea))                      # błąd względny w normie L2
        res['vel_L2'].append(np.sqrt(np.sum((un-ua)**2 + (vn-va)**2) / np.sum(ua**2 + va**2)))  # to samo dla wektora (u,v)
        res['eta_Linf'].append(np.abs(en - ea).max() / np.abs(ea).max())                         # błąd maksymalny / max|eta_analit|
    res['num'], res['ana'] = (en, un, vn), (ea, ua, va)
    res['eps'] = np.abs(ea).max() / P['H']                                                       # miara nieliniowości max|eta|/H
    return res

### Test samego rozwiązania analitycznego
Zanim porównamy z solwerem, sprawdzamy, że wzór jest poprawny: całkujemy te same równania Fouriera niezależnie (RK4, drobny krok) i porównujemy z $V\,\mathrm{diag}(\dots)V^{-1}\hat{\mathbf b}$. Dodatkowo sprawdzamy zachowanie masy ($\overline{\eta}=0$) i to, że przy $t=T$ rozwiązanie jest już bliskie stanowi stacjonarnemu.

In [ ]:
Qt, dxt, _ = make_Q(64, P['q0'])
ana_t = LinearAnalytic(Qt, dxt)

# niezależna całka ODE dla wszystkich modów Fouriera naraz (RK4)
T_test, n_rk = 20.0, 4000
h_rk = T_test / n_rk
Xh = np.zeros_like(ana_t.b)
rhs = lambda Y: np.einsum('xyij,xyj->xyi', ana_t.A, Y) + ana_t.b
for _ in range(n_rk):
    k1 = rhs(Xh); k2 = rhs(Xh + .5*h_rk*k1); k3 = rhs(Xh + .5*h_rk*k2); k4 = rhs(Xh + h_rk*k3)
    Xh = Xh + h_rk/6 * (k1 + 2*k2 + 2*k3 + k4)
eta_rk = np.fft.ifft2(Xh[..., 0]).real
eta_an, u_an, v_an = ana_t(T_test)
print(f"max|eta_analit - eta_RK4| / max|eta| = {np.abs(eta_an - eta_rk).max() / np.abs(eta_an).max():.2e}")

eta_T, u_T, v_T = ana_t(P['T'])
eta_s, u_s, v_s = ana_t.steady()
print(f"średnia eta (zachowanie masy)         = {eta_T.mean():.2e}")
print(f"|eta(T) - eta_stacjonarne| / max|eta|  = {np.abs(eta_T - eta_s).max() / np.abs(eta_s).max():.2e}   (T = {P['T']:.0f}, 1/r = {1/P['r']:.1f})")
print(f"miara nieliniowości eps = max|eta|/H   = {np.abs(eta_T).max() / P['H']:.3f}   (dla q0 = {P['q0']})")

### 2. Przypadek bazowy: numeryka kontra analityka
Dwa przypadki na tej samej siatce ($N=64$, $\Delta t=0.1$ – jak w symulacji powyżej): oryginalna amplituda $q_0$ oraz $q_0/10$. Jeśli różnica w drugim przypadku jest wyraźnie mniejsza (względnie), to różnica w pierwszym wynika z nieliniowości, a nie z błędu schematu.

In [ ]:
base = {}
for q_amp in (P['q0'], P['q0'] / 10):
    try:
        base[q_amp] = run_and_compare(64, 0.1, q_amp)
    except AssertionError as e:
        print(f"q0 = {q_amp}: solwer przerwał ({e})")

fig, axs = pyplot.subplots(len(base), 4, figsize=(15, 3.6 * len(base)), squeeze=False)
for row, (q_amp, res) in zip(axs, base.items()):
    en, ea = res['num'][0], res['ana'][0]
    vm = max(np.abs(en).max(), np.abs(ea).max())
    for ax, fld, ttl, kw in ((row[0], en, 'numeryka  $\\eta$', dict(vmin=-vm, vmax=vm)),
                              (row[1], ea, 'analityka (liniowa)  $\\eta$', dict(vmin=-vm, vmax=vm)),
                              (row[2], en - ea, 'różnica  num − anal', dict(vmin=-np.abs(en-ea).max(), vmax=np.abs(en-ea).max()))):
        im = ax.imshow(fld.T, origin='lower', cmap='RdBu', **kw)
        ax.set_title(ttl); pyplot.colorbar(im, ax=ax, fraction=.046)
    row[0].set_ylabel(f"$q_0$ = {q_amp:g}\n$\\epsilon$ = {res['eps']:.2f}")
    # przekrój przez środek domeny: eta(x) i v(x) (v to prędkość styczna wiru na osi x)
    j = res['N'] // 2
    cut = lambda F: 0.5 * (F[:, j-1] + F[:, j])
    ax = row[3]
    ax.plot(res['X'], cut(en), 'C0', label='$\\eta$ num'); ax.plot(res['X'], cut(ea), 'k--', label='$\\eta$ anal')
    ax2 = ax.twinx()
    ax2.plot(res['X'], cut(res['num'][2]), 'C3', label='$v$ num'); ax2.plot(res['X'], cut(res['ana'][2]), 'k:', label='$v$ anal')
    ax.set_xlabel('x'); ax.set_title(f"przekrój y=0,  t={res['t'][-1]:.0f}")
    ax.legend(loc='lower right', fontsize=7); ax2.legend(loc='upper right', fontsize=7)
fig.tight_layout(); pyplot.show()

for q_amp, res in base.items():
    print(f"q0 = {q_amp:<6g} eps = {res['eps']:.3f} | błąd względny eta (L2) w chwilach {res['t']}:")
    print("          ", np.round(res['eta_L2'], 4), "| (u,v):", np.round(res['vel_L2'], 4))

### 3. Zależność od gęstości siatki i kroku czasowego
Zmieniamy $N$ (czyli $\Delta x=L/N$, przy stałej fizycznej domenie $L$, stałym $\sigma$ i $Q$) oraz $\Delta t$. Żeby wyizolować błąd *numeryczny*, używamy małej amplitudy $q_0/50$ – wtedy model nieliniowy jest bliski liniowemu i różnica względem analityki pochodzi głównie z dyskretyzacji.

Pomijamy kombinacje z $c\,\Delta t/\Delta x>0.7$ (fale grawitacyjne, $c=\sqrt{gH}$) – dla gradientu ciśnienia liczonego jawnie schemat przestaje być wtedy stabilny. Błędy to **błędy względne w normie L2** w chwili końcowej $t=T$.

In [ ]:
Q_LIN    = P['q0'] / 50
N_LIST   = [32, 64, 128]                  # można dopisać 256 (wolniej)
DT_LIST  = [0.5, 0.25, 0.1, 0.05, 0.025]
CFL_MAX  = 0.7

sweep = {}
t_start = time.perf_counter()
for Ng in N_LIST:
    for dt_ in DT_LIST:
        cfl = c0 * dt_ / (P['L'] / Ng)
        if cfl > CFL_MAX:
            continue
        try:
            sweep[(Ng, dt_)] = run_and_compare(Ng, dt_, Q_LIN)
        except AssertionError as e:
            print(f"N={Ng}, dt={dt_}: przerwano ({e})")
print(f"gotowe w {time.perf_counter() - t_start:.0f} s")

print(f"\n{'N':>4} {'dx':>6} {'dt':>6} {'c*dt/dx':>8} {'eta L2':>10} {'eta Linf':>10} {'(u,v) L2':>10}")
for (Ng, dt_), res in sweep.items():
    print(f"{Ng:>4} {res['dx']:>6.3f} {dt_:>6.3f} {c0*dt_/res['dx']:>8.2f} "
          f"{res['eta_L2'][-1]:>10.2e} {res['eta_Linf'][-1]:>10.2e} {res['vel_L2'][-1]:>10.2e}")

In [ ]:
from matplotlib.colors import LogNorm

def table(key):
    E = np.full((len(N_LIST), len(DT_LIST)), np.nan)
    for i, Ng in enumerate(N_LIST):
        for j, dt_ in enumerate(DT_LIST):
            if (Ng, dt_) in sweep:
                E[i, j] = sweep[(Ng, dt_)][key][-1]
    return E

E_eta, E_vel = table('eta_L2'), table('vel_L2')

# --- mapy cieplne błędu (N x dt) ---
fig, axs = pyplot.subplots(1, 2, figsize=(12, 3.8))
for ax, E, ttl in ((axs[0], E_eta, 'błąd względny $\\eta$ (L2), $t=T$'), (axs[1], E_vel, 'błąd względny $(u,v)$ (L2), $t=T$')):
    im = ax.imshow(np.ma.masked_invalid(E), norm=LogNorm(), cmap='viridis_r', aspect='auto')
    for i in range(E.shape[0]):
        for j in range(E.shape[1]):
            ax.text(j, i, '—' if np.isnan(E[i, j]) else f"{E[i, j]:.1e}", ha='center', va='center', fontsize=8, color='w')
    ax.set_xticks(range(len(DT_LIST)), [str(d) for d in DT_LIST]); ax.set_xlabel('$\\Delta t$')
    ax.set_yticks(range(len(N_LIST)), [f"N={n}  (dx={P['L']/n:.2f})" for n in N_LIST])
    ax.set_title(ttl); pyplot.colorbar(im, ax=ax)
fig.tight_layout(); pyplot.show()

# --- wykresy zbieżności ---
fig, axs = pyplot.subplots(1, 2, figsize=(12, 4))
ax = axs[0]
for i, Ng in enumerate(N_LIST):
    ax.loglog(DT_LIST, E_eta[i], 'o-', label=f'N={Ng}')
ax.set_xlabel('$\\Delta t$'); ax.set_ylabel('błąd względny $\\eta$'); ax.set_title('zależność od kroku czasowego'); ax.grid(True, which='both', alpha=.3)
dref = np.array([DT_LIST[-1], DT_LIST[1]])
for p in (1, 2):
    ax.loglog(dref, np.nanmin(E_eta[:, 1]) * (dref / dref[-1])**p, 'k:', lw=.8); ax.text(dref[0], np.nanmin(E_eta[:, 1]) * (dref[0]/dref[-1])**p, f'  ~$\\Delta t^{p}$', fontsize=8)
ax.legend()

ax = axs[1]
dxs = np.array([P['L'] / n for n in N_LIST])
for j, dt_ in enumerate(DT_LIST):
    ax.loglog(dxs, E_eta[:, j], 'o-', label=f'$\\Delta t$={dt_}')
ax.set_xlabel('$\\Delta x$'); ax.set_ylabel('błąd względny $\\eta$'); ax.set_title('zależność od gęstości siatki'); ax.grid(True, which='both', alpha=.3)
xref = np.array([dxs.min(), dxs.max()])
for p in (1, 2):
    ax.loglog(xref, np.nanmin(E_eta) * 3 * (xref / xref[0])**p, 'k:', lw=.8); ax.text(xref[1], np.nanmin(E_eta) * 3 * (xref[1]/xref[0])**p, f'  ~$\\Delta x^{p}$', fontsize=8)
ax.legend(fontsize=8)
fig.tight_layout(); pyplot.show()

# --- obserwowany rząd zbieżności (kolejne pary) ---
print("rząd w czasie  p = log(e1/e2)/log(dt1/dt2)   (kolumny: kolejne pary dt)")
for i, Ng in enumerate(N_LIST):
    p = [np.log(E_eta[i, j] / E_eta[i, j+1]) / np.log(DT_LIST[j] / DT_LIST[j+1]) for j in range(len(DT_LIST) - 1)]
    print(f"  N={Ng:<4}", np.round(p, 2))
print("rząd w przestrzeni  p = log(e1/e2)/log(dx1/dx2)   (kolumny: kolejne pary N)")
for j, dt_ in enumerate(DT_LIST):
    p = [np.log(E_eta[i, j] / E_eta[i+1, j]) / np.log(dxs[i] / dxs[i+1]) for i in range(len(N_LIST) - 1)]
    print(f"  dt={dt_:<6}", np.round(p, 2))
print("(nan = pominięta kombinacja niestabilna; p bliskie 0 = błąd nasycił się, dominuje drugi składnik)")

**Jak czytać te wykresy.** Błąd całkowity to z grubsza suma wkładu przestrzennego (rośnie z $\Delta x$) i czasowego (rośnie z $\Delta t$). Wzdłuż wiersza (stałe $N$) zmniejszanie $\Delta t$ poprawia wynik tylko do pewnego momentu – potem błąd „nasyca się” na poziomie wyznaczonym przez siatkę. Analogicznie wzdłuż kolumny (stały $\Delta t$) zagęszczanie siatki pomaga do poziomu wyznaczonego przez krok czasowy. Rzędy zbieżności (tabela powyżej) mają sens tylko tam, gdzie błąd nie jest jeszcze nasycony.

In [ ]:
# --- ewolucja błędu w czasie dla różnych siatek (stały, mały dt) ---
DT_REF = 0.05
fig, axs = pyplot.subplots(1, 2, figsize=(12, 3.8), sharex=True)
for Ng in N_LIST:
    if (Ng, DT_REF) in sweep:
        res = sweep[(Ng, DT_REF)]
        axs[0].semilogy(res['t'], res['eta_L2'], 'o-', label=f'N={Ng}')
        axs[1].semilogy(res['t'], res['vel_L2'], 'o-', label=f'N={Ng}')
for ax, ttl in zip(axs, ('$\\eta$', '$(u,v)$')):
    ax.set_xlabel('t'); ax.set_title(f'błąd względny {ttl} w czasie ($\\Delta t$={DT_REF}, $q_0/50$)'); ax.grid(True, which='both', alpha=.3); ax.legend()
fig.tight_layout(); pyplot.show()

### 4. Wpływ nieliniowości (amplituda źródła $q_0$)
Stała siatka ($N=64$, $\Delta t=0.05$), rosnąca amplituda ujścia $q_0$. Miara nieliniowości: $\epsilon=\max|\eta|/H$ z rozwiązania analitycznego. Poziomą przerywaną linią zaznaczono błąd z reżimu liniowego dla tej samej siatki i kroku – to jest czysty błąd numeryczny (błąd względny nie zależy w nim od amplitudy). Linia rosnąca ponad nią to różnica między modelem nieliniowym a liniowym.

In [ ]:
Q_AMPS = [P['q0'] / 8, P['q0'] / 4, P['q0'] / 2, P['q0']]
nl = {}
for q_amp in Q_AMPS:
    try:
        nl[q_amp] = run_and_compare(64, 0.05, q_amp)
    except AssertionError as e:
        print(f"q0 = {q_amp:g}: solwer przerwał ({e})")

eps  = [nl[q]['eps'] for q in nl]
err  = [nl[q]['eta_L2'][-1] for q in nl]
errv = [nl[q]['vel_L2'][-1] for q in nl]

fig, ax = pyplot.subplots(figsize=(6.5, 4.2))
ax.loglog(eps, err, 'o-', label='$\\eta$'); ax.loglog(eps, errv, 's-', label='$(u,v)$')
if (64, 0.05) in sweep:
    ax.axhline(sweep[(64, 0.05)]['eta_L2'][-1], color='C0', ls='--', lw=.8, label='błąd numeryczny $\\eta$ (reżim liniowy)')
    ax.axhline(sweep[(64, 0.05)]['vel_L2'][-1], color='C1', ls='--', lw=.8, label='błąd numeryczny $(u,v)$ (reżim liniowy)')
ax.loglog(eps, np.array(eps) * (err[-1] / eps[-1]), 'k:', lw=.8, label='~ $\\epsilon$')
ax.set_xlabel('$\\epsilon=\\max|\\eta|/H$'); ax.set_ylabel('błąd względny względem analityki liniowej'); ax.grid(True, which='both', alpha=.3); ax.legend(fontsize=7)
fig.tight_layout(); pyplot.show()
for q_amp, e1, e2, e3 in zip(nl, eps, err, errv):
    print(f"q0 = {q_amp:<8g} eps = {e1:.3f}   błąd eta = {e2:.2e}   błąd (u,v) = {e3:.2e}")

### Podsumowanie metody
* **Rozwiązanie referencyjne:** dokładne rozwiązanie zlinearyzowanego układu (Fourier + rozkład własny macierzy $3\times3$ dla każdej fali), zweryfikowane niezależnym całkowaniem RK4.
* **Błąd:** różnica numeryka − analityka, mierzona jako błąd względny w normie L2 (oraz $L_\infty$) dla $\eta$ i wektora $(u,v)$, w funkcji $N$, $\Delta t$, czasu i amplitudy $q_0$.
* **Interpretacja:** w reżimie małych amplitud błąd maleje przy zagęszczaniu siatki i skracaniu kroku aż do nasycenia; dla dużych amplitud błąd względem rozwiązania *liniowego* rośnie z $\epsilon$ i nie jest błędem schematu, tylko odzwierciedla fizykę nieliniową, której wzór analityczny nie zawiera.